# Demo 6: Multi-Agent Orchestration

This demo demonstrates multi-agent orchestration: the supervisor-worker pattern where a supervisor agent routes questions to specialized agents (billing, shipping, technical). We register UC functions as tools, create a multi-agent system, and validate routing with separate test cells.

> **Prerequisite**: Complete Demo 5 (Building Agents) first to understand single-agent tool selection before moving to multi-agent systems.

In [0]:
%sql
-- SETUP: Create catalog, schema, sample data, and UC functions
-- We create a dedicated catalog/schema for this demo with a
-- support_tickets table for multi-agent routing scenarios.
-- UC functions registered here serve as specialized agent tools.

CREATE CATALOG IF NOT EXISTS module5a_demo6;
CREATE SCHEMA IF NOT EXISTS module5a_demo6.agent_systems;

-- Sample support tickets table for multi-agent routing demos
CREATE OR REPLACE TABLE module5a_demo6.agent_systems.support_tickets (
  ticket_id    STRING NOT NULL,
  customer_id  STRING NOT NULL,
  category     STRING,
  priority     STRING,
  subject      STRING,
  description  STRING,
  status       STRING,
  created_at   TIMESTAMP
);

INSERT INTO module5a_demo6.agent_systems.support_tickets VALUES
('TKT-001', 'CUST-001', 'billing',   'high',   'Overcharged on order ORD-001', 'I was charged $249.99 but the product was on sale for $199.99', 'open', '2026-09-25 08:00:00'),
('TKT-002', 'CUST-002', 'shipping', 'medium', 'Order ORD-003 not delivered',   'My order from Sept 20 has not arrived yet',                     'open', '2026-09-25 09:30:00'),
('TKT-003', 'CUST-003', 'technical','high',   'Cannot access my account',     'I keep getting a 403 error when trying to log in',             'open', '2026-09-25 10:15:00'),
('TKT-004', 'CUST-001', 'general',   'low',    'Product recommendation',      'Can you recommend accessories for my recent purchase?',        'open', '2026-09-25 11:00:00'),
('TKT-005', 'CUST-002', 'billing',   'medium', 'Discount not applied',        'My premium discount was not applied to order ORD-008',         'open', '2026-09-25 11:45:00');

-- UC functions as specialized agent tools
CREATE OR REPLACE FUNCTION module5a_demo6.agent_systems.get_ticket(ticket_id STRING)
RETURNS STRING
COMMENT 'Returns the subject and description of a support ticket'
RETURN (SELECT max(concat_ws(' | ', subject, description))
       FROM module5a_demo6.agent_systems.support_tickets
       WHERE ticket_id = get_ticket.ticket_id);

CREATE OR REPLACE FUNCTION module5a_demo6.agent_systems.count_tickets_by_category(cat STRING)
RETURNS INT
COMMENT 'Returns the number of tickets in a given category'
RETURN SELECT count(*) FROM module5a_demo6.agent_systems.support_tickets
       WHERE category = count_tickets_by_category.cat;

CREATE OR REPLACE FUNCTION module5a_demo6.agent_systems.get_priority(ticket_id STRING)
RETURNS STRING
COMMENT 'Returns the priority of a support ticket'
RETURN (SELECT max(priority) FROM module5a_demo6.agent_systems.support_tickets
       WHERE ticket_id = get_priority.ticket_id);

SELECT * FROM module5a_demo6.agent_systems.support_tickets ORDER BY ticket_id;

ticket_id,customer_id,category,priority,subject,description,status,created_at
TKT-001,CUST-001,billing,high,Overcharged on order ORD-001,I was charged .99 but the product was on sale for .99,open,2026-09-25T08:00:00.000Z
TKT-002,CUST-002,shipping,medium,Order ORD-003 not delivered,My order from Sept 20 has not arrived yet,open,2026-09-25T09:30:00.000Z
TKT-003,CUST-003,technical,high,Cannot access my account,I keep getting a 403 error when trying to log in,open,2026-09-25T10:15:00.000Z
TKT-004,CUST-001,general,low,Product recommendation,Can you recommend accessories for my recent purchase?,open,2026-09-25T11:00:00.000Z
TKT-005,CUST-002,billing,medium,Discount not applied,My premium discount was not applied to order ORD-008,open,2026-09-25T11:45:00.000Z


## 5.2 : Supervisor-Worker (Router-Delegate) Pattern

### Concepts
The supervisor-worker pattern is the most common multi-agent architecture:

* **Supervisor**: Receives the user's question, decides which specialist to delegate to, and synthesizes the final response.
* **Workers**: Specialized agents, each with its own system prompt, tools, and domain expertise.
* **Router**: The supervisor's routing decision can be LLM-driven (the model decides) or rule-based (deterministic).

**Flow**: User -> Supervisor (routes) -> Worker (executes) -> Supervisor (synthesizes) -> User

**Common worker specializations**:
* Data Agent: Queries tables, runs SQL, returns structured data
* Research Agent: Searches knowledge base, retrieves documents
* General Agent: Handles FAQs, small talk, general questions

In [0]:
# Setup: Install agents SDK and configure Databricks as the LLM provider
# The OpenAI Agents SDK works on Databricks by pointing it to the Databricks serving endpoint.

%pip install openai-agents --quiet 2>&1 | tail -1

import os
from openai import AsyncOpenAI
from agents import Agent, Runner, function_tool
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from databricks.sdk.core import Config

# Get Databricks credentials (automatic on serverless compute)
cfg = Config()
headers = cfg.authenticate()
token = headers['Authorization'].replace('Bearer ', '')
os.environ["OPENAI_API_KEY"] = token
os.environ["OPENAI_BASE_URL"] = f"{cfg.host}/serving-endpoints"

# Databricks uses Chat Completions API (not Responses API)
# So we configure the agents SDK with OpenAIChatCompletionsModel
client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ["OPENAI_BASE_URL"])
MODEL = "databricks-meta-llama-3-3-70b-instruct"

# Create @function_tool tools wrapping our UC functions
@function_tool
def get_ticket_info(ticket_id: str) -> str:
    """Returns the subject and description of a support ticket"""
    return spark.sql(f"SELECT module5a_demo6.agent_systems.get_ticket('{ticket_id}')").collect()[0][0]

@function_tool
def get_ticket_priority(ticket_id: str) -> str:
    """Returns the priority of a support ticket (high, medium, low)"""
    return spark.sql(f"SELECT module5a_demo6.agent_systems.get_priority('{ticket_id}')").collect()[0][0]

@function_tool
def count_tickets(category: str) -> int:
    """Returns the number of tickets in a given category"""
    return spark.sql(f"SELECT module5a_demo6.agent_systems.count_tickets_by_category('{category}')").collect()[0][0]

print("=== Setup Complete ===")
print(f"Model: {MODEL}")
print(f"Tools: get_ticket_info, get_ticket_priority, count_tickets")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
=== Setup Complete ===
Model: databricks-meta-llama-3-3-70b-instruct
Tools: get_ticket_info, get_ticket_priority, count_tickets


In [0]:
# Create 3 specialist Agent objects + 1 supervisor Agent with handoffs
# Each specialist is a REAL Agent object with its own instructions, model, and tools.
# The supervisor hands off to specialists using the handoffs parameter.

model = OpenAIChatCompletionsModel(model=MODEL, openai_client=client)

# Specialist 1: Billing Agent
billing_agent = Agent(
    name="BillingAgent",
    instructions="You are a billing specialist. Handle overcharges, discounts, payments, invoices. Use your tools to look up ticket info and priority. Be concise.",
    model=model,
    tools=[get_ticket_info, get_ticket_priority]
)

# Specialist 2: Shipping Agent
shipping_agent = Agent(
    name="ShippingAgent",
    instructions="You are a shipping specialist. Handle delivery delays, lost packages, tracking. Use your tools to look up ticket info and count tickets. Be concise.",
    model=model,
    tools=[get_ticket_info, count_tickets]
)

# Specialist 3: Technical Agent
technical_agent = Agent(
    name="TechnicalAgent",
    instructions="You are a technical specialist. Handle login errors, system access, 403/500 errors. Use your tools to look up ticket info and priority. Be concise.",
    model=model,
    tools=[get_ticket_info, get_ticket_priority]
)

# Supervisor Agent - hands off to specialists
supervisor = Agent(
    name="Supervisor",
    instructions="You are a supervisor. Route billing questions to BillingAgent, shipping to ShippingAgent, technical to TechnicalAgent.",
    model=model,
    handoffs=[billing_agent, shipping_agent, technical_agent]
)

print("=== Multi-Agent System Created ===")
print(f"Supervisor: {supervisor.name}")
print(f"  Handoffs: {[a.name for a in supervisor.handoffs]}")
print(f"BillingAgent tools: {[t.name for t in billing_agent.tools]}")
print(f"ShippingAgent tools: {[t.name for t in shipping_agent.tools]}")
print(f"TechnicalAgent tools: {[t.name for t in technical_agent.tools]}")

=== Multi-Agent System Created ===
Supervisor: Supervisor
  Handoffs: ['BillingAgent', 'ShippingAgent', 'TechnicalAgent']
BillingAgent tools: ['get_ticket_info', 'get_ticket_priority']
ShippingAgent tools: ['get_ticket_info', 'count_tickets']
TechnicalAgent tools: ['get_ticket_info', 'get_ticket_priority']


In [0]:
# Test Q1: Billing question
# The supervisor receives the question and hands off to BillingAgent.
# BillingAgent uses its tools to look up the ticket and responds.

print("USER: I was overcharged on my order ORD-001, check ticket TKT-001")
print()

result = await Runner.run(supervisor, "I was overcharged on my order ORD-001, check ticket TKT-001")

print(f"Routed to: {result.last_agent.name}")
print(f"Response: {result.final_output}")
print()

if result.last_agent.name == "BillingAgent":
    print("✅ VALIDATION: Supervisor -> BillingAgent -> Tool used -> Response")
else:
    print(f"❌ VALIDATION: Routed to {result.last_agent.name}, expected BillingAgent")

USER: I was overcharged on my order ORD-001, check ticket TKT-001



[non-fatal] Tracing client error 401. Response data is redacted.


Routed to: BillingAgent
Response: It seems there might be a discrepancy in the sale price and the charged amount for order ORD-001, as mentioned in ticket TKT-001. However, according to the ticket description, both the sale price and the charged amount are $0.99, which appears to be the same. Could you provide more details or clarify the issue you're experiencing with your order?

✅ VALIDATION: Supervisor -> BillingAgent -> Tool used -> Response


In [0]:
# Test Q2: Shipping question
# The supervisor receives the question and hands off to ShippingAgent.
# ShippingAgent uses its tools to look up the ticket and responds.

print("USER: My package from order ORD-003 hasn't arrived, check ticket TKT-002")
print()

result = await Runner.run(supervisor, "My package from order ORD-003 hasn't arrived, check ticket TKT-002")

print(f"Routed to: {result.last_agent.name}")
print(f"Response: {result.final_output}")
print()

if result.last_agent.name == "ShippingAgent":
    print("✅ VALIDATION: Supervisor -> ShippingAgent -> Tool used -> Response")
else:
    print(f"❌ VALIDATION: Routed to {result.last_agent.name}, expected ShippingAgent")

USER: My package from order ORD-003 hasn't arrived, check ticket TKT-002



[non-fatal] Tracing client error 401. Response data is redacted.


Routed to: ShippingAgent
Response: I've checked on the status of your order ORD-003, and according to the ticket TKT-002, it hasn't been delivered yet. You placed the order on September 20, and it's still pending arrival. I apologize for the inconvenience. Would you like me to look into this further or provide information on how to track your package?

✅ VALIDATION: Supervisor -> ShippingAgent -> Tool used -> Response


In [0]:
# Test Q3: Technical question
# The supervisor receives the question and hands off to TechnicalAgent.
# TechnicalAgent uses its tools to look up the ticket and responds.

print("USER: I can't log in, getting a 403 error, check ticket TKT-003")
print()

result = await Runner.run(supervisor, "I can't log in, getting a 403 error, check ticket TKT-003")

print(f"Routed to: {result.last_agent.name}")
print(f"Response: {result.final_output}")
print()

if result.last_agent.name == "TechnicalAgent":
    print("✅ VALIDATION: Supervisor -> TechnicalAgent -> Tool used -> Response")
else:
    print(f"❌ VALIDATION: Routed to {result.last_agent.name}, expected TechnicalAgent")

USER: I can't log in, getting a 403 error, check ticket TKT-003



[non-fatal] Tracing client error 401. Response data is redacted.


Routed to: TechnicalAgent
Response: A 403 error typically indicates that you don't have permission to access the resource. Given the high priority of your ticket, I recommend checking your account status and ensuring that your login credentials are correct. If the issue persists, it may be related to a system or configuration problem that needs to be addressed by the technical team. They will investigate and resolve the issue as soon as possible.

✅ VALIDATION: Supervisor -> TechnicalAgent -> Tool used -> Response


## 5.6 : Agent Bricks Development Lifecycle

### Concepts
The Agent Bricks lifecycle follows a specify -> configure -> improve loop:

1. **Specify**: Define the agent's purpose, knowledge sources, and tools
   - Upload documents or point to UC volumes/tables
   - Define what the agent should and should not do

2. **Configure & Evaluate**: Test the agent on your actual data
   - The platform automatically evaluates quality using built-in judges
   - Review accuracy, groundedness, and safety scores
   - Iterate on instructions and configuration

3. **Deploy**: Publish the agent as a serving endpoint
   - Automatically versioned and governed by Unity Catalog
   - Monitor with inference tables

4. **Continuously Improve**: Use evaluation results to improve
   - Add examples to improve quality
   - Update knowledge sources
   - Re-evaluate and redeploy

> The entire lifecycle is governed: UC permissions, audit logs, and lineage tracking apply at every step.

In [0]:
# CLEANUP: Drop all resources created in this demo
print("Dropping UC functions...")
for fn in ['get_ticket', 'count_tickets_by_category', 'get_priority']:
    spark.sql(f"DROP FUNCTION IF EXISTS module5a_demo6.agent_systems.{fn}")
print("  Functions dropped")

print("Dropping schema and catalog...")
spark.sql("DROP SCHEMA IF EXISTS module5a_demo6.agent_systems CASCADE")
spark.sql("DROP CATALOG IF EXISTS module5a_demo6 CASCADE")
print("  Schema and catalog dropped")

print("\nCleanup complete!")